# iLSU-T (WhisperX) → GASLT Training (Colab)

Este notebook entrena **GASLT** usando iLSU-T (videos + WhisperX) a través del export SignJoey-compatible generado por `lsu-pria`.

## Nota importante (recursos)
GASLT suele usar *grafito/similitud* entre ejemplos. Para que sea viable en Colab con iLSU-T, este notebook incluye un paso de **cálculo de similitudes** con embeddings de texto.

Si te explota RAM/tiempo:
- reduci `MAX_SEGMENTS_PER_EPISODE` (ej: 50–200), o
- setea `LIMIT` (ej: 10000) para exportar menos filas.


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Clone lsu-pria + install deps
# Si tu repo es privado o no está en GitHub, podés subirlo como ZIP a Colab y descomprimirlo en /content/lsu-pria.
LSU_PRIA_REPO_URL = ""  # @param {type:"string"}

!rm -rf /content/lsu-pria
if LSU_PRIA_REPO_URL.strip():
    !git clone -q "$LSU_PRIA_REPO_URL" /content/lsu-pria
else:
    print("LSU_PRIA_REPO_URL vacío: subí el repo como ZIP o setá la URL.")

%cd /content/lsu-pria

!python3 -m pip install -q -U pip
!python3 -m pip install -q -r requirements.txt

# Extras
!python3 -m pip install -q pandas pyyaml torchtext sacrebleu sentence-transformers scikit-learn


In [ ]:
#@title Config — Paths + Parameters
from pathlib import Path

# TODO: ajustá este path al lugar real en tu Drive
ILSUT_ROOT = "/content/drive/MyDrive/ilsut_extracted"  # @param {type:"string"}

WORK_ROOT = "/content/drive/MyDrive/lsu-pria-runs/whisperx_slt_gaslt"  # @param {type:"string"}
SOURCES = ["source2", "source3"]

MIN_WORDS = 1
MAX_WORDS = 60
MAX_CHARS = 240
MIN_DUR_MS = 700
MAX_DUR_MS = 25000
MAX_SEGMENTS_PER_EPISODE = 120  # GASLT: poner algo finito para que el paso de similitud sea viable

SAMPLE_FPS = 6.0
MAX_FRAMES = 48
PREPROCESS = True

LIMIT = 0

Path(WORK_ROOT).mkdir(parents=True, exist_ok=True)
print('WORK_ROOT:', WORK_ROOT)


In [ ]:
#@title Build WhisperX → SLT subset (clips + target_text)
from pathlib import Path

subset_dir = Path(WORK_ROOT) / 'subset'

cmd = f"""python3 scripts/prepare_whisperx_slt_subset.py \
  --root \"{ILSUT_ROOT}\" \
  --work-dir \"{subset_dir}\" \
  --sources {' '.join(SOURCES)} \
  --min-words {MIN_WORDS} \
  --max-words {MAX_WORDS} \
  --max-chars {MAX_CHARS} \
  --min-duration-ms {MIN_DUR_MS} \
  --max-duration-ms {MAX_DUR_MS} \
  --max-segments-per-episode {MAX_SEGMENTS_PER_EPISODE} \
  --export-clips \
  --clip-ext .mp4
"""
print(cmd)
!{cmd}


In [ ]:
#@title Export SignJoey bundle
from pathlib import Path

export_dir = Path(WORK_ROOT) / 'export'

cmd = f"""python3 scripts/export_ilsut_slt_dataset.py \
  --subset-dir \"{subset_dir}\" \
  --out-dir \"{export_dir}\" \
  --mode features \
  --sample-fps {SAMPLE_FPS} \
  --max-frames {MAX_FRAMES} \
  --clip-ext .mp4 \
  --backend neccam_slt \
  --limit {LIMIT} \
  {'--preprocess' if PREPROCESS else ''}
"""
print(cmd)
!{cmd}


## Prepare GASLT similarity files

Genera:
- `name_to_video_id.json`
- `cos_sim.pkl`

A partir de los textos de entrenamiento.


In [ ]:
#@title Build cos_sim.pkl (text-embedding cosine similarity)
import gzip
import pickle
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

bundle_dir = Path(WORK_ROOT) / 'export' / 'backend' / 'neccam_slt' / 'data' / 'ilsut'
train_path = bundle_dir / 'ilsut.train'

out_sim_dir = bundle_dir / 'sim'
out_sim_dir.mkdir(parents=True, exist_ok=True)

def load_gzip_pickle(p: Path):
    with gzip.open(p, 'rb') as f:
        return pickle.load(f)

train = load_gzip_pickle(train_path)
sentences = [ex['txt'] for ex in train]

# Map names to indices (video_id proxy)
name_to_video_id = {str(i): int(i) for i in range(len(sentences))}

model_name = 'distiluse-base-multilingual-cased-v1'
model = SentenceTransformer(model_name)
emb = model.encode(sentences, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

# Cosine similarity matrix (N x N). OJO: esto puede ser grande.
cos_sim = emb @ emb.T

(out_sim_dir / 'name_to_video_id.json').write_text(json.dumps(name_to_video_id, indent=2), encoding='utf-8')
with open(out_sim_dir / 'cos_sim.pkl', 'wb') as f:
    pickle.dump(cos_sim.astype(np.float32), f)

print('Wrote:', out_sim_dir / 'name_to_video_id.json')
print('Wrote:', out_sim_dir / 'cos_sim.pkl', 'shape=', cos_sim.shape)


## Train GASLT


In [ ]:
#@title Clone GASLT repo
!rm -rf /content/gaslt
!git clone -q https://github.com/YinAoXiong/GASLT.git /content/gaslt
%cd /content/gaslt

!python3 -m pip install -q -U pip
!python3 -m pip install -q -e . || true

!ls -la
!find . -maxdepth 3 -type f \( -name "sign.yaml" -o -name "*.yaml" -o -name "*.yml" \) | head -n 50


In [ ]:
#@title Copy dataset bundle into GASLT data folder
from pathlib import Path
import shutil

SRC_ILSUT = Path(WORK_ROOT) / 'export' / 'backend' / 'neccam_slt' / 'data' / 'ilsut'
DST_DATA = Path('/content/gaslt/data/ilsut')
DST_DATA.mkdir(parents=True, exist_ok=True)

for fn in ['ilsut.train','ilsut.val','ilsut.test']:
    shutil.copy2(SRC_ILSUT / fn, DST_DATA / fn)

# Copy sim files
(DST_DATA / 'sim').mkdir(parents=True, exist_ok=True)
shutil.copy2(SRC_ILSUT / 'sim' / 'name_to_video_id.json', DST_DATA / 'sim' / 'name_to_video_id.json')
shutil.copy2(SRC_ILSUT / 'sim' / 'cos_sim.pkl', DST_DATA / 'sim' / 'cos_sim.pkl')

!ls -la /content/gaslt/data/ilsut | head -n 80
!ls -la /content/gaslt/data/ilsut/sim | head -n 50


In [ ]:
#@title Generate config (patch sign.yaml)
import time
from pathlib import Path

repo_root = Path('/content/gaslt')
configs_dir = repo_root / 'configs'
configs_dir.mkdir(parents=True, exist_ok=True)

candidates = [repo_root / 'configs' / 'sign.yaml', repo_root / 'configs' / 'sign.yml']
base_cfg = next((p for p in candidates if p.exists()), None)
if base_cfg is None:
    found = list(repo_root.glob('**/sign.yaml')) + list(repo_root.glob('**/sign.yml'))
    base_cfg = found[0] if found else None
if base_cfg is None:
    raise RuntimeError('No base config found (expected configs/sign.yaml).')

cfg_txt = base_cfg.read_text(encoding='utf-8')
feature_size = 105

def set_scalar(key: str, value: str) -> None:
    global cfg_txt
    lines = cfg_txt.splitlines()
    out = []
    replaced = False
    for ln in lines:
        if ln.lstrip().startswith(key + ':'):
            indent = ln[:len(ln) - len(ln.lstrip())]
            out.append(f"{indent}{key}: {value}")
            replaced = True
        else:
            out.append(ln)
    if not replaced:
        out.append(f"{key}: {value}")
    cfg_txt = "\n".join(out)

def set_section_scalar(section: str, key: str, value: str) -> None:
    global cfg_txt
    lines = cfg_txt.splitlines()
    out = []
    in_section = False
    replaced = False
    for ln in lines:
        if ln.startswith(section + ':'):
            in_section = True
            out.append(ln)
            continue
        if in_section and ln and not ln.startswith(' '):
            in_section = False
        if in_section and ln.lstrip().startswith(key + ':'):
            indent = ln[:len(ln) - len(ln.lstrip())]
            out.append(f"{indent}{key}: {value}")
            replaced = True
        else:
            out.append(ln)
    if not replaced:
        out2 = []
        inserted = False
        for ln in out:
            out2.append(ln)
            if ln.startswith(section + ':') and not inserted:
                out2.append(f"    {key}: {value}")
                inserted = True
        out = out2
    cfg_txt = "\n".join(out)

set_scalar('name', f"ilsut_gaslt_{int(time.time())}")

# data
set_section_scalar('data', 'data_path', repr(str(repo_root / 'data')))
set_section_scalar('data', 'train', 'ilsut/ilsut.train')
set_section_scalar('data', 'dev', 'ilsut/ilsut.val')
set_section_scalar('data', 'test', 'ilsut/ilsut.test')
set_section_scalar('data', 'feature_size', str(int(feature_size)))
set_section_scalar('data', 'level', 'word')

# training
set_section_scalar('training', 'use_cuda', 'true')
set_section_scalar('training', 'overwrite', 'true')

# GASLT-specific keys (best effort; si el repo usa nombres distintos, ajustá aquí)
set_scalar('cos_sim_path', repr('data/ilsut/sim/cos_sim.pkl'))
set_scalar('name_to_video_id_path', repr('data/ilsut/sim/name_to_video_id.json'))

out_cfg = configs_dir / 'ilsut_gaslt.yaml'
out_cfg.write_text(cfg_txt + "\n", encoding='utf-8')
print('Wrote:', out_cfg)
!sed -n '1,160p' /content/gaslt/configs/ilsut_gaslt.yaml


In [ ]:
#@title Train
%cd /content/gaslt
!python3 -c "import signjoey; print('signjoey ok:', signjoey.__file__)" || true

# Algunos forks usan --gpu_id; otros detectan automático.
!python3 -m signjoey train configs/ilsut_gaslt.yaml || python3 -m signjoey train configs/ilsut_gaslt.yaml --gpu_id 0
